In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
from google.cloud import bigquery

client = bigquery.Client(project="flooit-analytics-project")

In [3]:
import pandas as pd

In [4]:
day_0 = pd.read_csv('day_0.csv', parse_dates = ['day_0'])

print(day_0.shape)
print(day_0.dtypes)
day_0.head()

(4319, 2)
user_pseudo_id            object
day_0             datetime64[ns]
dtype: object


,user_pseudo_id,day_0
0,1814A35D096333518F94B02DDFE3BFEC,2018-06-15
1,4F8458200DBF0AD3A4B095CA35EEA47C,2018-06-15
2,6B41795D5E5B7339E007330941C9E201,2018-06-15
3,0FB42D5FFB79A9DE147873753BF7A664,2018-07-15
4,61C77C8D9E7F92524289DA7E9D4786BF,2018-07-15


In [5]:
query = """WITH params AS (SELECT user_pseudo_id,
                       event_date,
                       event_timestamp,
                       event_name,
                       param.key AS parameter_name,
                       COALESCE(param.value.string_value,
                                CAST(param.value.int_value    AS STRING),
                                CAST(param.value.float_value  AS STRING),
                                CAST(param.value.double_value AS STRING)) AS param_value
                FROM `firebase-public-project.analytics_153293282.events_*` CROSS JOIN UNNEST(event_params) AS param
                WHERE event_name IN ('in_app_purchase', 'ad_reward')),

     pivoted AS (SELECT user_pseudo_id, event_date, event_timestamp, event_name,
                     MAX(IF(parameter_name = 'product_id', param_value, NULL)) AS product_id,
                     MAX(IF(parameter_name = 'price', param_value, NULL)) AS price,
                     MAX(IF(parameter_name = 'currency', param_value, NULL)) AS currency,
                     MAX(IF(parameter_name = 'validated', param_value, NULL)) AS validated,
                     MAX(IF(parameter_name = 'type', param_value, NULL)) AS type,
                     MAX(IF(parameter_name = 'value', param_value, NULL)) AS value
                 FROM params
                 GROUP BY user_pseudo_id, event_date, event_timestamp, event_name)

SELECT user_pseudo_id, event_date, event_timestamp, event_name, product_id, SAFE_CAST(price AS FLOAT64)/1000000 AS price, currency, validated, type, SAFE_CAST(value AS FLOAT64) AS value
FROM pivoted
ORDER BY user_pseudo_id, event_timestamp"""

mon = client.query(query).to_dataframe()
print(mon.shape)

(1939, 10)


In [6]:
mon.to_csv("monetization_events.csv", index=False)

In [7]:
mon = pd.read_csv('monetization_events.csv', parse_dates = ['event_date'])

print(mon.shape)
print(mon.dtypes)
display(mon.head(15))

display(mon.notna().sum())

display(mon[(mon.event_name == 'ad_reward') & (mon.type.isna())]['value'].describe())

(1939, 10)
user_pseudo_id             object
event_date         datetime64[ns]
event_timestamp             int64
event_name                 object
product_id                 object
price                     float64
currency                   object
validated                 float64
type                       object
value                     float64
dtype: object


,user_pseudo_id,event_date,event_timestamp,event_name,product_id,price,currency,validated,type,value
0,00AE0CA4117376AE083FD6AEA744CE88,2018-09-02,1535943983925000,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
1,00AE0CA4117376AE083FD6AEA744CE88,2018-09-02,1535944080985000,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
2,014EE05605C94D797CA52C355638C967,2018-08-13,1534198373295000,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
3,014EE05605C94D797CA52C355638C967,2018-08-13,1534198445681031,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
4,014EE05605C94D797CA52C355638C967,2018-08-13,1534198451217008,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
5,025FF377B4148748AF5743A54761EA15,2018-06-26,1530010973066014,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
6,025FF377B4148748AF5743A54761EA15,2018-06-26,1530011082853096,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
7,025FF377B4148748AF5743A54761EA15,2018-06-26,1530011118403136,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
8,025FF377B4148748AF5743A54761EA15,2018-06-29,1530267199262014,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
9,026E9E99CF63F4AD31203F6B2E01949D,2018-07-12,1531454060475001,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0


,0
user_pseudo_id,1939
event_date,1939
event_timestamp,1939
event_name,1939
product_id,27
price,27
currency,27
validated,16
type,1878
value,1939


,value
count,34.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


In [8]:
cohort_share = pd.merge(day_0, mon, on='user_pseudo_id', how='inner')
display(cohort_share)

cohort_share_pct = round(100.0 * cohort_share.shape[0] / mon.shape[0], 2)
print(cohort_share_pct)

print(cohort_share.groupby('event_name').size())

,user_pseudo_id,day_0,event_date,event_timestamp,event_name,product_id,price,currency,validated,type,value
0,23D89EE594C105BFA999295B38C80B2B,2018-06-17,2018-06-25,1529969258260015,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
1,23D89EE594C105BFA999295B38C80B2B,2018-06-17,2018-06-25,1529969378807111,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
2,CEA79F13743A84BC8ABA238AB7D2851A,2018-07-29,2018-07-29,1532887607312172,ad_reward,NaN,NaN,NaN,NaN,NaN,1.0
3,C8B06F76EE9091107007C19B08C7834E,2018-07-07,2018-07-07,1531013957345055,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
4,C8B06F76EE9091107007C19B08C7834E,2018-07-07,2018-07-07,1531014293124152,ad_reward,NaN,NaN,NaN,NaN,Steps,2.0
...,...,...,...,...,...,...,...,...,...,...,...
505,22430BD1EDFC8E2E980313F10148A53E,2018-06-29,2018-07-04,1530725088446000,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
506,A16DE718D8908FFC01DD297FEA600583,2018-09-17,2018-09-17,1537242133741001,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
507,1C0AF5475798F990F25A06C5A162E8DF,2018-07-30,2018-07-30,1532963380668009,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0
508,1C0AF5475798F990F25A06C5A162E8DF,2018-07-30,2018-07-30,1532984265537000,ad_reward,NaN,NaN,NaN,NaN,Steps,5.0


26.3
event_name
ad_reward          493
in_app_purchase     17
dtype: int64


In [9]:
query = """WITH params AS (SELECT user_pseudo_id,
                       event_timestamp,
                       param.key AS parameter_name,
                       COALESCE(param.value.string_value,
                                CAST(param.value.int_value    AS STRING),
                                CAST(param.value.float_value  AS STRING),
                                CAST(param.value.double_value AS STRING)) AS param_value
                FROM `firebase-public-project.analytics_153293282.events_*` CROSS JOIN UNNEST(event_params) AS param
                WHERE event_name = 'post_score'),

     pivoted AS (SELECT user_pseudo_id, event_timestamp,
                     MAX(IF(parameter_name = 'level', param_value, NULL)) AS level,
                     MAX(IF(parameter_name = 'score', param_value, NULL)) AS score
                 FROM params
                 GROUP BY user_pseudo_id, event_timestamp)

SELECT level, score, CASE WHEN SAFE_CAST(level AS INT64) = 0 THEN 'quickplay' ELSE 'level_mode' END AS mode, COUNT(*) AS n_posts
FROM pivoted
GROUP BY level, score, mode
ORDER BY SAFE_CAST(level AS INT64), SAFE_CAST(score AS INT64)"""

ps = client.query(query).to_dataframe()
print(ps.shape)

(312, 4)


In [10]:
ps.to_csv("post_score_distribution.csv", index=False)

In [11]:
ps = pd.read_csv('post_score_distribution.csv')

print(ps.shape)
print(ps.dtypes)
ps.head()

(312, 4)
level       int64
score       int64
mode       object
n_posts     int64
dtype: object


,level,score,mode,n_posts
0,0,0,quickplay,56174
1,0,1,quickplay,52321
2,0,2,quickplay,43179
3,0,3,quickplay,29123
4,0,4,quickplay,15832


In [12]:
ps['n_posts'].sum()

np.int64(242039)

In [13]:
query = """WITH user_window AS (SELECT user_pseudo_id, IF(LOGICAL_OR(event_name = 'first_open'), 'in_window', 'pre_window') AS window_group
                     FROM `firebase-public-project.analytics_153293282.events_*`
                     GROUP BY user_pseudo_id),

     labeled AS (SELECT e.user_pseudo_id, event_date, event_name, window_group
                 FROM `firebase-public-project.analytics_153293282.events_*` AS e JOIN user_window AS uw
                 ON e.user_pseudo_id = uw.user_pseudo_id),

     arm_dates AS (SELECT event_date AS period, window_group, event_name, COUNT(*) AS n_events, COUNT(DISTINCT user_pseudo_id) AS n_users
                   FROM labeled
                   WHERE event_date IN ('20180618','20180619','20180620','20180621','20180622','20180623','20180624','20180625','20180626','20180627','20180628','20180629','20180630','20180701')
                   GROUP BY period, window_group, event_name),

     arm_block AS (SELECT 'stable_block' AS period, window_group, event_name, COUNT(*) AS n_events, COUNT(DISTINCT user_pseudo_id) AS n_users
                   FROM labeled
                   WHERE event_date >= '20180702'
                   GROUP BY window_group, event_name)

SELECT * FROM arm_dates
UNION ALL
SELECT * FROM arm_block"""

sf = client.query(query).to_dataframe()
print(sf.shape)
print(sf.groupby(['period','window_group','event_name']).size().max())

(853, 5)
1


In [14]:
sf.to_csv("spike_forensics.csv", index=False)

In [15]:
sf = pd.read_csv('spike_forensics.csv')

print(sf.shape)
print(sf.dtypes)
sf.head()

(853, 5)
period          object
window_group    object
event_name      object
n_events         int64
n_users          int64
dtype: object


,period,window_group,event_name,n_events,n_users
0,stable_block,pre_window,level_retry_quickplay,18044,1457
1,stable_block,in_window,level_end_quickplay,41897,1919
2,stable_block,pre_window,level_reset_quickplay,83464,1561
3,stable_block,pre_window,screen_view,1493336,6966
4,stable_block,in_window,select_content,26139,3617
